In [ ]:
import os
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from football_ai.data import list_available_dataset_keys, load_xy, split_dataset_key

DATA_DIR = os.path.join('..', 'data', 'spadl_data')

def build_models() -> dict[str, object]:
    """Notebook-specific model set (broader than library default)."""
    return {
        'Decision Tree': DecisionTreeClassifier(random_state=42, class_weight='balanced'),
        'Extra Trees': ExtraTreesClassifier(
            n_estimators=200, random_state=42, n_jobs=-1, class_weight='balanced_subsample'
        ),
        'Gradient Boosting': GradientBoostingClassifier(random_state=42),
        'MLP': Pipeline([
            ('scaler', StandardScaler()),
            ('clf', MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=300, random_state=42))
        ]),
        'Lasso': Pipeline([
            ('scaler', StandardScaler()),
            ('clf', LogisticRegression(
                penalty='l1', solver='liblinear', class_weight='balanced', max_iter=1000, random_state=42
            ))
        ]),
    }

## 1. Discover available league-season datasets

List all available `features_*.h5` / `labels_*.h5` pairs generated in `data/spadl_data`.

In [ ]:
available_dataset_keys = list_available_dataset_keys(DATA_DIR)

print('Total available datasets:', len(available_dataset_keys))
print('First 10 dataset keys + parsed league/season:')
for key in available_dataset_keys[:10]:
    league, season = split_dataset_key(key)
    print('-', key, '=>', league, '|', season)

Total available datasets: 75
First 10 dataset keys + parsed league/season:
- 1_bundesliga_2015_2016 => 1 bundesliga | 2015/2016
- 1_bundesliga_2023_2024 => 1 bundesliga | 2023/2024
- african_cup_of_nations_2023 => african cup of nations | 2023
- champions_league_1970_1971 => champions league | 1970/1971
- champions_league_1971_1972 => champions league | 1971/1972
- champions_league_1972_1973 => champions league | 1972/1973
- champions_league_1999_2000 => champions league | 1999/2000
- champions_league_2003_2004 => champions league | 2003/2004
- champions_league_2004_2005 => champions league | 2004/2005
- champions_league_2006_2007 => champions league | 2006/2007


## 2. Configure training dataset and target

Choose one training league-season and one target (`scores` or `concedes`). Models are trained in the next step.

In [8]:
# Configure training
TARGET_COL = 'scores'  # or 'concedes'
TRAIN_DATASET_KEY = 'serie_a_2015_2016'  # <- set this to one key from available_dataset_keys

if TRAIN_DATASET_KEY not in available_dataset_keys:
    raise ValueError(
        f'TRAIN_DATASET_KEY={TRAIN_DATASET_KEY!r} not found. Pick one from available_dataset_keys.'
    )

X_train, y_train = load_xy(TRAIN_DATASET_KEY, target_col=TARGET_COL, data_dir=DATA_DIR)
train_feature_cols = list(X_train.columns)

models = build_models()
trained_models = {}
for model_name, model in models.items():
    model.fit(X_train, y_train)
    trained_models[model_name] = model
    print(f'Trained {model_name} on {TRAIN_DATASET_KEY}')

print('\nTarget:', TARGET_COL)
print('X_train shape:', X_train.shape)
print('Positive rate:', float(y_train.mean()))

Trained Decision Tree on serie_a_2015_2016
Trained Extra Trees on serie_a_2015_2016
Trained Gradient Boosting on serie_a_2015_2016
Trained MLP on serie_a_2015_2016


c:\Users\anma10\PycharmProjects\supervised-learning-statsbomb\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\anma10\PycharmProjects\supervised-learning-statsbomb\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


Trained Lasso on serie_a_2015_2016

Target: scores
X_train shape: (761736, 36)
Positive rate: 0.010357919279120326


## 3. Test on multiple other league-seasons (one table per model)

Each model gets its own table with separate `league` and `season` columns, plus accuracy/precision/recall/f1.

In [ ]:
# Leave empty to test on all datasets except the training one
TEST_DATASET_KEYS = [
    # 'serie_a_1986_1987',
    # 'premier_league_2015_2016',
]

if TEST_DATASET_KEYS:
    test_dataset_keys = TEST_DATASET_KEYS
else:
    test_dataset_keys = [k for k in available_dataset_keys if k != TRAIN_DATASET_KEY]

results_tables_by_model = {}

for model_name, model in trained_models.items():
    rows = []
    for test_key in test_dataset_keys:
        league, season = split_dataset_key(test_key)
        try:
            X_test, y_test = load_xy(test_key, target_col=TARGET_COL, data_dir=DATA_DIR)
            X_test_aligned = X_test.reindex(columns=train_feature_cols, fill_value=0)

            y_pred = model.predict(X_test_aligned)

            rows.append({
                'league': league,
                'season': season,
                'tested_league_year': test_key,
                'accuracy': accuracy_score(y_test, y_pred),
                'precision': precision_score(y_test, y_pred, zero_division=0),
                'recall': recall_score(y_test, y_pred, zero_division=0),
                'f1': f1_score(y_test, y_pred, zero_division=0),
                'n_samples': int(len(y_test)),
            })
        except Exception as e:
            rows.append({
                'league': league,
                'season': season,
                'tested_league_year': test_key,
                'accuracy': None,
                'precision': None,
                'recall': None,
                'f1': None,
                'n_samples': 0,
                'error': str(e),
            })

    table = pd.DataFrame(rows)
    base_cols = ['league', 'season', 'tested_league_year', 'accuracy', 'precision', 'recall', 'f1']
    other_cols = [c for c in table.columns if c not in base_cols]
    table = table[base_cols + other_cols].sort_values(['league', 'season']).reset_index(drop=True)
    results_tables_by_model[model_name] = table

    print(f'\n=== {model_name} ===')
    display(table)
    print('Evaluated datasets:', len(table))


=== Decision Tree ===


,league,season,tested_league_year,accuracy,precision,recall,f1,n_samples
0,1 bundesliga,2015/2016,1_bundesliga_2015_2016,0.980883,0.158787,0.145608,0.151912,596336
1,1 bundesliga,2023/2024,1_bundesliga_2023_2024,0.979380,0.128733,0.133976,0.131303,80212
2,african cup of nations,2023,african_cup_of_nations_2023,0.981307,0.155054,0.154146,0.154599,92441
3,champions league,1970/1971,champions_league_1970_1971,0.977876,0.096774,0.150000,0.117647,2034
4,champions league,1971/1972,champions_league_1971_1972,0.984607,0.125000,0.125000,0.125000,1819
...,...,...,...,...,...,...,...,...
69,uefa europa league,1988/1989,uefa_europa_league_1988_1989,0.979671,0.129870,0.158730,0.142857,5903
70,uefa women s euro,2022,uefa_women_s_euro_2022,0.980522,0.164756,0.156889,0.160727,61660
71,uefa women s euro,2025,uefa_women_s_euro_2025,0.977605,0.201094,0.160656,0.178615,60370
72,women s world cup,2019,women_s_world_cup_2019,0.981068,0.158460,0.152586,0.155468,101575


Evaluated datasets: 74

=== Extra Trees ===


,league,season,tested_league_year,accuracy,precision,recall,f1,n_samples
0,1 bundesliga,2015/2016,1_bundesliga_2015_2016,0.988978,0.966030,0.064889,0.121609,596336
1,1 bundesliga,2023/2024,1_bundesliga_2023_2024,0.988979,1.000000,0.052519,0.099796,80212
2,african cup of nations,2023,african_cup_of_nations_2023,0.989766,0.975904,0.079024,0.146209,92441
3,champions league,1970/1971,champions_league_1970_1971,0.990659,1.000000,0.050000,0.095238,2034
4,champions league,1971/1972,champions_league_1971_1972,0.992303,1.000000,0.125000,0.222222,1819
...,...,...,...,...,...,...,...,...
69,uefa europa league,1988/1989,uefa_europa_league_1988_1989,0.989836,0.800000,0.063492,0.117647,5903
70,uefa women s euro,2022,uefa_women_s_euro_2022,0.988810,0.921569,0.064120,0.119898,61660
71,uefa women s euro,2025,uefa_women_s_euro_2025,0.985655,0.980392,0.054645,0.103520,60370
72,women s world cup,2019,women_s_world_cup_2019,0.989210,0.921053,0.060345,0.113269,101575


Evaluated datasets: 74

=== Gradient Boosting ===


...

Evaluated datasets: 74

=== MLP ===


,league,season,tested_league_year,accuracy,precision,recall,f1,n_samples
0,1 bundesliga,2015/2016,1_bundesliga_2015_2016,0.989482,0.841328,0.130063,0.225296,596336
1,1 bundesliga,2023/2024,1_bundesliga_2023_2024,0.989416,0.795775,0.121115,0.210233,80212
2,african cup of nations,2023,african_cup_of_nations_2023,0.990167,0.829545,0.142439,0.243131,92441
3,champions league,1970/1971,champions_league_1970_1971,0.991150,1.000000,0.100000,0.181818,2034
4,champions league,1971/1972,champions_league_1971_1972,0.992303,1.000000,0.125000,0.222222,1819
...,...,...,...,...,...,...,...,...
69,uefa europa league,1988/1989,uefa_europa_league_1988_1989,0.990344,0.714286,0.158730,0.259740,5903
70,uefa women s euro,2022,uefa_women_s_euro_2022,0.989199,0.759690,0.133697,0.227378,61660
71,uefa women s euro,2025,uefa_women_s_euro_2025,0.986202,0.750000,0.134426,0.227989,60370
72,women s world cup,2019,women_s_world_cup_2019,0.989535,0.751295,0.125000,0.214339,101575


Evaluated datasets: 74

=== Lasso ===


,league,season,tested_league_year,accuracy,precision,recall,f1,n_samples
0,1 bundesliga,2015/2016,1_bundesliga_2015_2016,0.712555,0.029909,0.745864,0.057512,596336
1,1 bundesliga,2023/2024,1_bundesliga_2023_2024,0.678128,0.026523,0.747053,0.051227,80212
2,african cup of nations,2023,african_cup_of_nations_2023,0.719810,0.028399,0.730732,0.054674,92441
3,champions league,1970/1971,champions_league_1970_1971,0.695182,0.019231,0.600000,0.037267,2034
4,champions league,1971/1972,champions_league_1971_1972,0.680594,0.023609,0.875000,0.045977,1819
...,...,...,...,...,...,...,...,...
69,uefa europa league,1988/1989,uefa_europa_league_1988_1989,0.714552,0.028488,0.777778,0.054964,5903
70,uefa women s euro,2022,uefa_women_s_euro_2022,0.711985,0.031996,0.793997,0.061512,61660
71,uefa women s euro,2025,uefa_women_s_euro_2025,0.689349,0.035757,0.750820,0.068263,60370
72,women s world cup,2019,women_s_world_cup_2019,0.700231,0.028433,0.761207,0.054819,101575


Evaluated datasets: 74


## 4. Combined comparison table (all models)

Stack all model tables into one table so you can compare `accuracy`, `precision`, `recall`, and `f1` side-by-side.

In [15]:
comparison_rows = []
for model_name, table in results_tables_by_model.items():
    t = table.copy()
    t['model'] = model_name
    comparison_rows.append(t)

comparison_table = pd.concat(comparison_rows, ignore_index=True)

# Keep requested metrics visible and ordered
priority_cols = [
    'model',
    'league',
    'season',
    'accuracy',
    'precision',
    'recall',
    'f1',
    'tested_league_year',
    'n_samples',
]
remaining_cols = [c for c in comparison_table.columns if c not in priority_cols]
comparison_table = comparison_table[priority_cols + remaining_cols]

comparison_table = comparison_table.sort_values(
    ['model', 'league', 'season'], ascending=[True, True, True]
).reset_index(drop=True)

display(comparison_table)
print('Rows in combined comparison table:', len(comparison_table))

,model,league,season,accuracy,precision,recall,f1,tested_league_year,n_samples
0,Decision Tree,1 bundesliga,2015/2016,0.980883,0.158787,0.145608,0.151912,1_bundesliga_2015_2016,596336
1,Decision Tree,1 bundesliga,2023/2024,0.979380,0.128733,0.133976,0.131303,1_bundesliga_2023_2024,80212
2,Decision Tree,african cup of nations,2023,0.981307,0.155054,0.154146,0.154599,african_cup_of_nations_2023,92441
3,Decision Tree,champions league,1970/1971,0.977876,0.096774,0.150000,0.117647,champions_league_1970_1971,2034
4,Decision Tree,champions league,1971/1972,0.984607,0.125000,0.125000,0.125000,champions_league_1971_1972,1819
...,...,...,...,...,...,...,...,...,...
365,MLP,uefa europa league,1988/1989,0.990344,0.714286,0.158730,0.259740,uefa_europa_league_1988_1989,5903
366,MLP,uefa women s euro,2022,0.989199,0.759690,0.133697,0.227378,uefa_women_s_euro_2022,61660
367,MLP,uefa women s euro,2025,0.986202,0.750000,0.134426,0.227989,uefa_women_s_euro_2025,60370
368,MLP,women s world cup,2019,0.989535,0.751295,0.125000,0.214339,women_s_world_cup_2019,101575


Rows in combined comparison table: 370


## 5. Optional: save CSV files

Save each model table and also the combined comparison table.

In [16]:
saved_paths = []
for model_name, table in results_tables_by_model.items():
    model_slug = model_name.lower().replace(' ', '_')
    out_path = os.path.join(
        DATA_DIR,
        f'eval_{TARGET_COL}_train_{TRAIN_DATASET_KEY}_{model_slug}.csv',
    )
    table.to_csv(out_path, index=False)
    saved_paths.append(out_path)

combined_out_path = os.path.join(
    DATA_DIR,
    f'eval_{TARGET_COL}_train_{TRAIN_DATASET_KEY}_all_models.csv',
)
comparison_table.to_csv(combined_out_path, index=False)
saved_paths.append(combined_out_path)

print('Saved files:')
for p in saved_paths:
    print('-', p)

Saved files:
- data\spadl_data\eval_scores_train_serie_a_2015_2016_decision_tree.csv
- data\spadl_data\eval_scores_train_serie_a_2015_2016_extra_trees.csv
- data\spadl_data\eval_scores_train_serie_a_2015_2016_gradient_boosting.csv
- data\spadl_data\eval_scores_train_serie_a_2015_2016_mlp.csv
- data\spadl_data\eval_scores_train_serie_a_2015_2016_lasso.csv
- data\spadl_data\eval_scores_train_serie_a_2015_2016_all_models.csv


## 6. Notes

- `league` and `season` are separate columns in every model table.
- `comparison_table` includes all models and all metrics (`accuracy`, `precision`, `recall`, `f1`).
- `results_tables_by_model` is a dictionary: key = model name, value = model-specific metrics table.
- Included models: Decision Tree, Extra Trees, Gradient Boosting, MLP, Lasso (L1-regularized logistic model).